# FLUPS 3: process the scans

Template on 260605, runs `run03` and `run04` (401 delays, -1.5 to 6.5 ps). Needs the two
calibration files written by the sibling templates: the t0 curve of `flups_1_t0` and the PCF of
`flups_2_pcf` (run them first). Both are on the detected wavelength `wl_u`.

Steps: read, compare the runs, despike, reject bad data, average, subtract the pre-t0 spectrum,
shift t0, correct with the PCF, convert to the fluorescence wavelength `wl_f`, save.

The interactive tool (`ScanInspector`) is preceded by a **decisions** cell and followed by the
cell that uses its result, which calls `print_decisions()`: paste the printout into the decisions
cell. With no clicks, "Run all" gives the result recorded there; after clicking, paste and re-run
before saving. Choices (scientific or taste) are marked `# choice`.

Needs `%matplotlib widget` (JupyterLab or VS Code, with `ipympl`).

In [ ]:
%matplotlib widget
from pathlib import Path

import numpy as np

from edyx.calib import interp_calibration, read_calibration
from edyx.fana import despike_median, fluo_wl, pcf_correct
from edyx.io import read_dat, stack_traces, write_dat
from edyx.plot import plot_maps
from edyx.tana import avg_scans, shift_trace, subtract_pre_t0
from pyspan import Overlay, ScanInspector
from pyspan.decisions import rmsd_vs_median

## Parameters

In [ ]:
data_dir = Path("../../../edyx/examples/260605")  # edyx example data; in the lab: Path(".")
fnames = sorted(data_dir.glob("260605_run0[34].dat"))  # or an explicit list
date = data_dir.resolve().name  # or enter a date manually
t0_file = Path("260605_t0_PBBO.toml")  # written by flups_1_t0
pcf_file = Path("260605_pcf_dyes.toml")  # written by flups_2_pcf
gate = 1340  # nm, gate wavelength of the experiment

despike_size = (15, 5)  # choice: median filter size (wavelengths, delays)
despike_threshold = 8  # counts, choice: see "Despike"
t_pre = -0.8  # ps, choice: delays t < t_pre are before time zero
t0_offset = 0.0  # ps, choice: residual t0, e.g. from a Raman line

out_dir = Path(".")
out_root = "run0304"
names = [f.name for f in fnames]
names

## Read

Axes of `stack`: (file, wl_u, delay).

In [ ]:
t, wl_u, stack = stack_traces([read_dat(f) for f in fnames])
stack.shape

## Compare the runs

First vs last run, to see a degradation of the sample or a drift. The raw data.

In [ ]:
Overlay({names[0]: (t, wl_u, stack[0]), names[-1]: (t, wl_u, stack[-1])}, mode="kinetics").show()

## Despike

`despike_median` replaces the points that differ from the local median by more than the
threshold, one scan at a time. Recipe for the threshold, computed once and pasted above: about
2 x (p95 - p5) of a region without signal, here `wl_u` 480-520 nm and delays before -0.8 ps,
pooled over the scans. The commented lines give 8.0.

The map shows one scan: raw, despiked, and what was removed. The overlay compares the
despiked scans.

In [ ]:
# bg = stack[:, (wl_u > 480) & (wl_u < 520)][..., t < -0.8].compressed()
# 2 * np.subtract(*np.percentile(bg, [95, 5]))  # -> 8.0, rounded to the literal above
despiked = np.array([despike_median(s, despike_size, threshold=despike_threshold) for s in stack])
int((despiked != stack).sum())  # replaced points

In [ ]:
i = 0  # choice: scan shown in the check
fig, axes = plot_maps(t, wl_u, [stack[i], despiked[i], stack[i] - despiked[i]],
                      titles=["raw", "despiked", "removed"], ncols=3, vmin=-10, vmax=10)

In [ ]:
Overlay.from_stack(t, wl_u, despiked, labels=names).show()

## Reject data

RMSD of each scan against the median of the scans. With 2 files the median is the mean, so both
files get the same RMSD and the threshold cannot single out one file.

In [ ]:
rmsd = rmsd_vs_median(despiked, axis=0)

In [ ]:
# Paste the output of print_decisions() here. Format: (file, wl, t) indices, None = all,
# (start, stop) = range, stop excluded. E.g. [(1, None, (250, None))]
rejections = []
threshold = None  # None: no threshold

In [ ]:
insp = ScanInspector(t, wl_u, despiked, labels=names, rejections=rejections,
                     rmsd=rmsd, threshold=threshold)
insp.show()

In [ ]:
insp.print_decisions()
data = insp.masked()

## Average and subtract the pre-t0 spectrum

The acquisition already subtracts the background. Against a drift during a scan, the pre-t0
spectrum of each scan can be subtracted before averaging (optional, a choice): see the
commented line.

In [ ]:
avg = avg_scans(data)
# avg = avg_scans(subtract_pre_t0(t, data, t_pre))  # optional: per scan, against a drift
avg_sub = subtract_pre_t0(t, avg, t_pre)  # choice: pre-t0 spectrum of the average

fig, axes = plot_maps(t, wl_u, [avg, avg_sub], titles=["average", "pre-t0 subtracted"],
                      vmin=-9, vmax=9)
for ax in axes.flat:
    ax.set_ylim(250, 450)

## Shift t0

The t0 curve is on `wl_u`, like the data: `interp_calibration` evaluates it at the measured
wavelengths. It is reliable only where the t0 dye emits (about 283-346 nm for this curve) and
extrapolated elsewhere. `shift_trace` moves the signal at `t0[i]` to delay 0 and keeps only the
delays valid for every wavelength. `t0_offset` is a number, up to you.

In [ ]:
t0 = interp_calibration(read_calibration(t0_file), wl_u)
t_s, shifted = shift_trace(t, avg_sub, t0)
t_s = t_s - t0_offset

fig, axes = plot_maps(t, wl_u, [avg_sub], titles=["t0 curve"], vmin=-3, vmax=3)
axes[0, 0].plot(t0, wl_u, color="k", lw=0.5)
axes[0, 0].set_ylim(250, 450)
fig, axes = plot_maps(t_s, wl_u, [shifted], titles=["t0 corrected"], vmin=-3, vmax=3)
axes[0, 0].axvline(0, color="k", lw=0.5)
axes[0, 0].set_ylim(250, 450)

## PCF

`pcf_correct` interpolates the PCF on `wl_u` (a calibration, like t0) and multiplies. Wavelengths
outside the range of the PCF become **masked** points: the PCF is finite only 285-425 nm here.

In [ ]:
s_pcf = pcf_correct(wl_u, shifted, read_calibration(pcf_file), outside="nan")
fig, axes = plot_maps(t_s, wl_u, [shifted, s_pcf], titles=["t0 corrected", "PCF corrected"],
                      vmin=-5, vmax=5)
for ax in axes.flat:
    ax.set_ylim(250, 450)

## Fluorescence axis

`wl_f = fluo_wl(wl_u, gate)`. The signal is a density per unit `wl_u`; per unit `wl_f` it is
multiplied by `d wl_u / d wl_f`. Without and with the PCF, on `wl_f`.

In [ ]:
wl_f = fluo_wl(wl_u, gate)
# d wl_f / d wl_u = wl_f**2 / wl_u**2: a density per wl_u times wl_u**2 / wl_f**2 is per wl_f
jac = (wl_u**2 / wl_f**2)[:, np.newaxis]
s_f = s_pcf * jac
s_nopcf = shifted * jac

In [ ]:
Overlay({"no PCF": (t_s, wl_f, s_nopcf), "PCF": (t_s, wl_f, s_f)},
        mode="spectra", ylabel="counts / nm").show()

## Save

Rows where the PCF is undefined are fully masked: they are cropped, so the file has no empty
rows. The remaining masked points (rejected data) are written as NaN.

In [ ]:
ok = ~np.ma.getmaskarray(s_f).all(axis=1)  # wavelengths with a PCF
out_dir.mkdir(exist_ok=True)
header = (f"scans: {names}\n"
          f"despike: size {despike_size}, threshold {despike_threshold} counts\n"
          f"rejections: {rejections}, threshold {threshold}\n"
          f"t_pre: {t_pre} ps\n"
          f"t0 curve: {t0_file.name}, t0_offset: {t0_offset} ps\n"
          f"PCF: {pcf_file.name}\n"
          f"gate: {gate} nm\n"
          f"axis: wl_f, signal per unit wl_f")
out_file = out_dir / f"{date}_{out_root}.dat"
write_dat(out_file, t_s, wl_f[ok], s_f[ok], header=header)
print(f"{wl_f[ok][0]:.1f}-{wl_f[ok][-1]:.1f} nm, {ok.sum()} wavelengths x {len(t_s)} delays")

fig, axes = plot_maps(t_s, wl_f[ok], [s_f[ok]], vmin=-5, vmax=5)
fig.savefig(out_dir / f"{date}_{out_root}.png", dpi=300)